# Layer 3: two moods that can switch within a session

Layer 2 gave each session one hidden mood for all its tracks: *Engaged* (few skips) or *Browsing* (many skips). It explained most of the differences between sessions, but its predictive checks showed two things it could not reproduce:

- the **within-session gap**: in the real data a skip is followed by another skip more often than the session's own skip rate explains (0.10 in the real data, 0 in Layer 2),
- the **skip/play changes**: real sessions change from skip to play less often than Layer 2 predicts (2.74 against 3.95 per session). Skips come in runs.

In this notebook the mood can **switch** from one track to the next. Each track has its own hidden mood, and the mood of a track only depends on the mood of the track before: this is a hidden Markov model. If the switch probabilities are 0, we are back to Layer 2.

(the goal is to see whether adding switching beats Layer 2)

In [1]:
import json
import os
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import mood_switching as ms
import posterior as po
import session_types as st
from checks import (all_checks, average_log_score, load_sessions,
                    real_data_checks, split_users, to_matrix)

SEED = 2026
rng = np.random.default_rng(SEED)

N_DRAWS = 20_000     # importance sampling draws
DF = 4               # degrees of freedom of the Student-t proposal
N_POSTERIOR = 1000   # resampled posterior draws (predictive checks, test score)
N_STARTS = 10        # EM starting points

# Beta(2, 2) for theta_e, theta_b, p0, q_eb and q_be
# EM uses Beta(3, 3) to find the mode on the log-odds scale
PRIOR = ms.PRIOR
EM_PRIOR = po.log_odds_prior(PRIOR)

#make the dirs for the results
os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)

BLUE = "#00a10d"
ORANGE = "#aa3102"
GREY = "#a4a4a4"

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": GREY,
    "xtick.color": GREY,
    "ytick.color": GREY,
    "axes.labelcolor": GREY,
})

The model functions are in `mood_switching.py` and the general tools (importance sampling, Hessian, predictive checks) in `posterior.py`, used as they are in Layer 2.
We also import `session_types.py` (Layer 2) to compare the two models.

## 1. The data

We use the same sessions and the same training/test split as Layers 1 and 2 (split by user, seed 2026): 20% of the users are kept aside, only for the additional analysis at the end.

The results of Layers 1 and 2 are read from `results/layer1.json` and `results/layer2.json` for the comparisons.

In [2]:
sessions = load_sessions()
train, test = split_users(sessions, seed=SEED)
train_skips, train_mask = to_matrix(train)
test_skips, test_mask = to_matrix(test)
train_lengths = train_mask.sum(axis=1)

with open("results/layer1.json") as file:
    layer1 = json.load(file)
with open("results/layer2.json") as file:
    layer2 = json.load(file)

print(f"Training: {len(train_skips)} sessions, {train_mask.sum()} tracks, skip rate {train['skip'].mean():.4f}")
print(f"Test: {len(test_skips)} sessions, {test_mask.sum()} tracks")
print(f"Session length: {train_lengths.min()} to {train_lengths.max()} tracks, mean {train_lengths.mean():.2f}")

Training: 4314 sessions, 67122 tracks, skip rate 0.4185
Test: 1061 sessions, 16595 tracks
Session length: 10 to 20 tracks, mean 15.56


We recall the two statistics that Layer 2 could not reproduce, read from `results/layer2.json`:

In [3]:
layer2_checks = pd.DataFrame(layer2["predictive_checks"]).T
layer2_checks.loc[["within-session gap", "skip/play changes"],
                  ["real data", "simulated mean", "share at least as large"]].round(4)

,real data,simulated mean,share at least as large
within-session gap,0.1013,-0.0001,0.0
skip/play changes,2.7415,3.9486,1.0


The training data have 4314 sessions and 67 122 tracks, the same as in Layers 1 and 2 (skip rate around 0.42).
The test data is only used at the end for extra investigation.

The table shows what Layer 3 must fix: Layer 2 gives a within-session gap of 0 (real data 0.10) and too many skip/play changes (3.95 against 2.74).

## 2. The model

For session $i$ with $n_i$ tracks, $y_{it} = 1$ if track $t$ is skipped and $0$ if it is played.

- $z_{it} \in \{E, B\}$ is the hidden mood of track $t$ of session $i$ (Engaged or Browsing). It can now change from one track to the next.
- $p_0 = P(z_{i1} = E)$ is the probability that a session starts Engaged.
- $q_{EB} = P(z_{it} = B \mid z_{i,t-1} = E)$ is the probability to switch from Engaged to Browsing between two tracks, and $q_{BE}$ the probability to switch back.
- $\theta_E$ and $\theta_B$ are the skip rates of the two moods.

**The model**

$$z_{i1} \sim \text{Bernoulli}(p_0), \qquad P(z_{it} = j \mid z_{i,t-1} = k) = A_{kj}, \qquad y_{it} \mid z_{it} \sim \text{Bernoulli}(\theta_{z_{it}})$$

with the transition matrix (rows: mood before, columns: mood after)

$$A = \begin{pmatrix} 1 - q_{EB} & q_{EB} \\ q_{BE} & 1 - q_{BE} \end{pmatrix}$$

The moods form a Markov chain: the mood of a track only depends on the mood of the track before. Given its mood, a track is skipped independently of the others. This is a hidden Markov model.

**Likelihood.** We do not know the moods, so we add all the possible sequences of moods (law of total probability). There are $2^{n_i}$ of them, so we do not write them one by one: the **forward algorithm** adds them one track at a time. With $\alpha_t(j) = P(y_{i1}, \dots, y_{it}, z_{it} = j)$:

$$\alpha_1(j) = P(z_{i1} = j)\, P(y_{i1} \mid j), \qquad \alpha_t(j) = \Big(\sum_k \alpha_{t-1}(k)\, A_{kj}\Big)\, P(y_{it} \mid j), \qquad p(y_i) = \alpha_{n_i}(E) + \alpha_{n_i}(B)$$

The code works in logs, as in Layer 2, so long sessions do not underflow.

**The order of the tracks now matters**: in Layer 2 only $n_i$ and $s_i$ appeared. Here, two skips in a row are more likely than two skips far apart, because the mood tends to stay the same.

**Link with Layer 2**: if $q_{EB} = q_{BE} = 0$, nobody switches, the mood of the first track is the mood of the whole session, and we are back to Layer 2.

**Priors.**

| Parameter | Prior |  |
|---|---|---|
| $\theta_E, \theta_B$ | Beta(2, 2), with $\theta_E < \theta_B$ | Same as Layer 2. The order fixes the labels: "engaged" always means the mood with fewer skips |
| $p_0$ | Beta(2, 2) | Same as Layer 2 |
| $q_{EB}, q_{BE}$ | Beta(2, 2) | We have no strong knowledge about how often listeners switch |

Beta(2, 2) has the weight of 4 observations, while the data have about 63,000 pairs of neighbouring tracks, so the prior should matter very little (we will check it later).

As an example, we take $\theta_E = 0.1$, $\theta_B = 0.7$, $p_0 = 0.5$ and $q_{EB} = q_{BE} = 0.2$, and a session of two skips. By hand, we add the four sequences of moods:

| moods | P(moods) | P(two skips \| moods) | product |
|---|---|---|---|
| E, E | 0.5 × 0.8 = 0.4 | 0.1 × 0.1 = 0.01 | 0.004 |
| E, B | 0.5 × 0.2 = 0.1 | 0.1 × 0.7 = 0.07 | 0.007 |
| B, E | 0.5 × 0.2 = 0.1 | 0.7 × 0.1 = 0.07 | 0.007 |
| B, B | 0.5 × 0.8 = 0.4 | 0.7 × 0.7 = 0.49 | 0.196 |

So $p(y) = 0.214$, and $P(\text{Browsing at track 2} \mid \text{two skips}) = (0.007 + 0.196)/0.214 = 0.949$.

In [ ]:
example = {"theta_e": 0.1, "theta_b": 0.7, "p0": 0.5, "q_eb": 0.2, "q_be": 0.2}
two_skips = np.array([[1, 1]])
two_tracks = np.ones((1, 2), dtype=bool)

likelihood = np.exp(ms.log_likelihood(example, two_skips, two_tracks))
gamma, _ = ms.state_probabilities(example, two_skips, two_tracks)
p_browsing = gamma[0, 1, 1]   # session 0, track 2, Browsing
print(f"Likelihood of two skips: {likelihood:.4f} (by hand 0.214)")
print(f"P(Browsing at track 2 | two skips): {p_browsing:.4f} (by hand 0.949)")

Both numbers agree with the hand computation. With the same skip rates, Layer 2 gave 0.25 and 0.98: here the mood can switch, so two skips say a little less about the mood of each track.

We now check that the order of the tracks matters. The two sessions below both have 2 skips out of 4 tracks, but in the first one the skips are next to each other:

In [ ]:
same_counts = np.array([[1, 1, 0, 0],
                        [1, 0, 1, 0]])
four_tracks = np.ones((1, 4), dtype=bool)
pd.DataFrame({
    "Layer 2": [np.exp(st.log_likelihood(example, row[None], four_tracks)) for row in same_counts],
    "Layer 3": [np.exp(ms.log_likelihood(example, row[None], four_tracks)) for row in same_counts],
}, index=["skip, skip, play, play", "skip, play, skip, play"]).round(4)

Layer 2 gives the same probability to both sessions (only the number of skips counts). Layer 3 finds the first session about 1.6 times more likely, because the skips form a run, which is what the real data showed.

We now want to check that log_likelihood computes the right number, by computing it by brute force and comparing them, as in Layer 2.

We write every possible session of 1 to 4 tracks (30 sessions) in rows of 6 cells, with junk 1s after the end of each session to check that the mask really hides them. For each session we go through **every sequence of moods** ($2^{n_i}$ of them), multiply the probabilities directly (without logs and without the forward algorithm), and add them up.

Then, we check that it agrees with the code, which uses the forward algorithm in logs.

In [ ]:
from itertools import product

# Every session of 1 to 4 tracks:
blocks = []
tiny_lengths = []
for length in range(1, 5):
    codes = np.arange(2 ** length)
    digits = (codes[:, None] >> np.arange(6)) & 1
    # junk 1s after the end of the session
    blocks.append(np.where(np.arange(6) < length, digits, 1))
    tiny_lengths.append(np.full(2 ** length, length))
tiny_skips = np.vstack(blocks)
tiny_lengths = np.concatenate(tiny_lengths)
tiny_mask = np.arange(6) < tiny_lengths[:, None]

q = {"theta_e": 0.23, "theta_b": 0.81, "p0": 0.37, "q_eb": 0.12, "q_be": 0.31}
theta = {"E": q["theta_e"], "B": q["theta_b"]}
start = {"E": q["p0"], "B": 1 - q["p0"]}
switch = {("E", "E"): 1 - q["q_eb"], ("E", "B"): q["q_eb"],
          ("B", "E"): q["q_be"], ("B", "B"): 1 - q["q_be"]}

def brute_force(y):
    '''Add P(moods) * P(y | moods) over every sequence of moods'''
    total = 0
    for moods in product("EB", repeat=len(y)):
        p = start[moods[0]]
        for before, after in zip(moods[:-1], moods[1:]):
            p *= switch[(before, after)]
        for mood, skip in zip(moods, y):
            p *= theta[mood] if skip == 1 else 1 - theta[mood]
        total += p
    return total

direct = np.array([brute_force(row[:length]) for row, length in zip(tiny_skips, tiny_lengths)])
ours = np.array([np.exp(ms.log_likelihood(q, tiny_skips[[i]], tiny_mask[[i]]))
                 for i in range(len(tiny_skips))])
print(f"{len(direct)} sessions, largest difference: {np.abs(ours - direct).max():.1e}")
print(f"Sum over the 16 sessions of 4 tracks: {direct[tiny_lengths == 4].sum():.6f}")

The brute force agrees with the forward algorithm to a rounding error ($10^{-16}$), and the probabilities of all sessions of 4 tracks add up to 1.

We can therefore conclude that the model is a proper probability distribution, that the forward algorithm is right and that the mask works.

Last check: with $q_{EB} = q_{BE} = 0$, Layer 3 should give exactly the Layer 2 log-likelihood. We compare the two on the real training data, with the Layer 2 values of the skip rates and of $p_0$:

In [ ]:
no_switch = {**layer2["em_mode"], "q_eb": 0.0, "q_be": 0.0}
layer2_value = st.log_likelihood(layer2["em_mode"], train_skips, train_mask)
layer3_value = ms.log_likelihood(no_switch, train_skips, train_mask)
print(f"Layer 2: {layer2_value:.6f}")
print(f"Layer 3 without switching: {layer3_value:.6f}")
print(f"Difference: {abs(layer3_value - layer2_value):.1e}")

The two log-likelihoods are equal (up to rounding). So Layer 2 is exactly the special case of Layer 3 with no switching, and any improvement in Layer 3 can only come from the switches.